# Multi-Query Retrieval (Enterprise AI Pattern)

Multi-Query Retrieval is an **Advanced RAG** technique that improves retrieval quality by generating **multiple variations of the user's query** before searching the vector database.

It is commonly used in enterprise AI systems where users ask ambiguous, short, or poorly worded questions.

Typical interview questions include:

- What is Multi-Query Retrieval?
- Why not search using only the original query?
- How does Multi-Query Retrieval improve RAG?
- Explain its architecture.
- Multi-Query vs Query Rewriting?

---

# 1. What is Multi-Query Retrieval?

## Definition

Multi-Query Retrieval uses an **LLM** to generate multiple semantically different versions of the user's question. Each query is searched independently, and the results are merged before being sent to the LLM.

Instead of searching once, the system searches **multiple times** using different phrasings.

---

## Interview Answer

> Multi-Query Retrieval is an Advanced RAG technique where an LLM generates multiple semantically equivalent queries from the original user question. Each query retrieves relevant documents independently, and the combined results improve recall by reducing the chance of missing important information.

---

# 2. Why Do We Need Multi-Query Retrieval?

Suppose the HR document contains

```text
Annual Leave Policy
```

User asks

```text
PTO Rules
```

The retriever may not find the correct document.

---

If the LLM generates

```text
PTO Rules

↓

Paid Time Off Policy

↓

Annual Leave Policy

↓

Vacation Leave Policy
```

Now multiple searches are performed.

One of them will likely retrieve the correct document.

---

# 3. Problem with Traditional RAG

Traditional RAG

```text
User Question

↓

One Embedding

↓

One Search

↓

Top-K

↓

LLM
```

Only **one opportunity** to retrieve the correct document.

---

# 4. Multi-Query Retrieval Architecture

```text
                    User Question
                           │
                           ▼
                     AWS Bedrock /
                    Azure OpenAI
                (Generate Queries)
                           │
        ┌───────────┬───────────┬────────────┐
        ▼           ▼           ▼
     Query 1     Query 2     Query 3
        │           │           │
        ▼           ▼           ▼
    Retriever   Retriever   Retriever
        │           │           │
        ▼           ▼           ▼
 OpenSearch   OpenSearch   OpenSearch
   /Qdrant       /Qdrant      /Qdrant
        └───────────┬───────────┘
                    ▼
              Merge Results
                    ▼
              Remove Duplicates
                    ▼
             Top-K Documents
                    ▼
         AWS Bedrock / Azure OpenAI
                    ▼
                 Response
```

---

# AWS + Azure Components

| Layer | AWS | Azure |
|--------|------|--------|
| LLM | Bedrock | Azure OpenAI |
| Vector DB | OpenSearch / Qdrant | Azure AI Search / Qdrant |
| Storage | S3 | Blob Storage |
| Backend | ECS/EKS | Container Apps/AKS |

---

# 5. Example

Document

```text
Annual Leave Policy
```

---

User Question

```text
PTO Rules
```

---

LLM generates

```text
Query 1

PTO Rules
```

```text
Query 2

Paid Time Off Policy
```

```text
Query 3

Annual Leave Policy
```

```text
Query 4

Vacation Leave Policy
```

Each query performs a semantic search.

Results are merged.

The LLM receives a much richer context.

---

# 6. End-to-End Flow

```text
Question

↓

Generate Multiple Queries

↓

Embedding

↓

Vector Search

↓

Merge Documents

↓

Remove Duplicates

↓

Top-K

↓

Bedrock

↓

Answer
```

---

# 7. LangChain Example

```python
# ==========================================================
# STEP 1 : Create Bedrock LLM
#
# Purpose:
# The LLM generates multiple versions of the user's query.
# ==========================================================

from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model="anthropic.claude-3-5-sonnet-20241022-v2:0",
    region_name="us-east-1"
)


# ==========================================================
# STEP 2 : Create Vector Retriever
#
# Purpose:
# Connect to Qdrant/OpenSearch and create a retriever.
# ==========================================================

retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)


# ==========================================================
# STEP 3 : Create Multi Query Retriever
#
# Purpose:
# Generate multiple semantic variations of the query.
# ==========================================================

from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query = MultiQueryRetriever.from_llm(
    retriever=retriever,
    llm=llm
)


# ==========================================================
# STEP 4 : Retrieve Documents
#
# Purpose:
# Multiple searches are performed automatically.
# ==========================================================

docs = multi_query.invoke(
    "What is the PTO policy?"
)


# ==========================================================
# STEP 5 : Display Retrieved Documents
# ==========================================================

for doc in docs:
    print(doc.page_content)
```

> **Production Note:** `MultiQueryRetriever` automatically generates query variations, performs multiple retrievals, merges the results, and removes duplicates before returning the final document set.

---

# 8. Production Architecture

```text
User

↓

FastAPI

↓

JWT

↓

LangGraph

↓

Multi Query Retriever

↓

Generate 5 Queries

↓

OpenSearch

↓

Merge

↓

Reranker

↓

Context Compression

↓

Bedrock

↓

Redis

↓

Response
```

---

# 9. Advantages

✅ Better recall

✅ Finds documents with different terminology

✅ Handles ambiguous user questions

✅ Reduces missed documents

✅ Improves enterprise search quality

---

# 10. Disadvantages

❌ Multiple vector searches

❌ Higher latency

❌ More LLM tokens

❌ Increased cost

---

# 11. Best Practices

✅ Generate 3–5 query variations

✅ Combine with Hybrid Search

✅ Add Reranking

✅ Remove duplicate documents

✅ Apply metadata filters where appropriate

---

# 12. Common Mistakes

❌ Generating too many queries

❌ No deduplication

❌ No reranking

❌ Using it for very simple questions

---

# 13. Multi-Query vs Query Rewriting

| Query Rewriting | Multi-Query Retrieval |
|-----------------|-----------------------|
| Generates one improved query | Generates multiple queries |
| One retrieval | Multiple retrievals |
| Faster | Slightly slower |
| Lower cost | Higher cost |
| Better than original | Better recall |

Example

Original

```text
PTO Policy
```

---

Query Rewriting

```text
Annual Leave Policy
```

↓

One Search

---

Multi-Query

```text
PTO Policy

Paid Time Off Policy

Annual Leave Policy

Vacation Policy
```

↓

Four Searches

---

# 14. Real Enterprise Example

### Healthcare Assistant

Question

```text
Medicine for High Blood Sugar
```

LLM generates

```text
Diabetes Medication

Hyperglycemia Treatment

Type 2 Diabetes Drugs

Blood Sugar Medication
```

The retriever searches all four queries and merges the results.

---

### HR Assistant

Question

```text
PTO Policy
```

Generated queries

```text
Annual Leave Policy

Paid Time Off

Vacation Policy

Leave Rules
```

This significantly increases the chance of retrieving the correct HR document.

---

# 15. Multi-Query vs Hybrid Search

| Hybrid Search | Multi-Query Retrieval |
|---------------|-----------------------|
| One query | Multiple queries |
| Keyword + Vector | Multiple semantic searches |
| Improves precision | Improves recall |
| Faster | Slower |

In production, many enterprise systems use **both** together:

```text
Question

↓

Generate Multiple Queries

↓

Hybrid Search (BM25 + Vector)

↓

Merge

↓

Rerank

↓

LLM
```

---

# 16. Interview Questions

### Q1. Why Multi-Query Retrieval?

To improve recall by searching with multiple semantically equivalent queries.

---

### Q2. Why not use only Query Rewriting?

Query Rewriting produces only one improved query. If that query still misses the relevant document, retrieval fails. Multi-Query Retrieval performs several searches, increasing the probability of finding relevant information.

---

### Q3. Does Multi-Query Retrieval increase cost?

Yes.

It increases both:

- LLM token usage (for query generation)
- Vector database searches

---

### Q4. Can Multi-Query Retrieval be combined with Hybrid Search?

Yes.

This is a common production architecture.

---

### Q5. Where is Multi-Query Retrieval useful?

- HR Assistants
- Healthcare Assistants
- Banking AI
- Legal Search
- Enterprise Knowledge Bases

---

# 17. Comparison

| Traditional RAG | Query Rewriting | Multi-Query Retrieval |
|-----------------|-----------------|-----------------------|
| One query | One improved query | Multiple improved queries |
| One retrieval | One retrieval | Multiple retrievals |
| Lowest recall | Better recall | Highest recall |
| Lowest cost | Medium cost | Highest cost |

---

# 18. EPAM Senior Answer (3 Minutes)

> "Multi-Query Retrieval is an Advanced RAG technique that improves retrieval recall by generating multiple semantic variations of the user's question using an LLM. Instead of relying on a single embedding and a single search, the system creates several equivalent queries—for example, expanding 'PTO Policy' into 'Paid Time Off Policy', 'Annual Leave Policy', and 'Vacation Policy'. Each query retrieves documents independently from Amazon OpenSearch, Qdrant, or Azure AI Search, and the results are merged, deduplicated, and often reranked before being sent to Amazon Bedrock or Azure OpenAI. This approach reduces the risk of missing relevant documents when different terminology is used in enterprise content. In production, I commonly combine Multi-Query Retrieval with Hybrid Search, metadata filtering, reranking, and context compression to maximize retrieval quality while balancing latency and cost."